# Anomaly class detection on Network Traffic data (CICEVSE2024 dataset)

This notebook contains implementation for detecting category of anomaly in EVSE Network Traffic data. The implementation is done using open source Agno AI Agents. The implementation is done using open source Agno AI Agent. A multi-agent framework is implemented where each agent is responsible for a certain task. 4 Agents have been implemented for data loading & preprocessing, feature engineering, anomaly detection & report generation of the entire pipeline.
The detection agent uses a model LightGBM model for multiple class classification where the aim is to predict the anomaly class

## 1. Imports & Configuration

In [1]:
# !pip install agno openai lightgbm scikit-learn pandas numpy matplotlib seaborn joblib

In [2]:
import os
import glob
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc,
)

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools import tool

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


/Users/rasikasonar/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/rasikasonar/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
EVSE_A             = "Dataset_for_masters_thesis/CICEVSE2024_Dataset/Network_Traffic/EVSE-A/csv"
EVSE_B             = "Dataset_for_masters_thesis/CICEVSE2024_Dataset/Network_Traffic/EVSE-B/csv"
COMBINED_Directory = "Dataset_for_masters_thesis/CICEVSE2024_Dataset/Network_Traffic/Combined"
COMBINED_FILE_PATH = "Dataset_for_masters_thesis/CICEVSE2024_Dataset/Network_Traffic/Combined/Network_Traffic_Combined.csv"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


## 2. Shared data between the agents.

To share the databetween the agents, we would be making use of dataclasses. Model parameters, preprocessed dataframes, selected features, sequences will be shared among the agents using this dataclass

In [4]:
# State of the EV Charging/Idle has already been captured under a separate column State. 
# Every file in the Network Traffic data has State of EV prefixed. We will be stripping this column to 
# Have more concrete classes for model training.
STATE_PREFIXES = ("charging ", "idle ", "maliciousev ")

# Two filename-level inconsistencies have been found in the data. These two would be mapped to represent single value
ATTACK_TYPE_NORMALISATIONS = {
    "syn stealth": "syn stealth scan",   # 'charging syn stealth' is the same attack as the other captures' 'syn stealth scan'
    "portscan":    "port scan",          # 'portscan' (no space) is used in some files; canonicalise to 'port scan'
}

SCAN_FAMILY = {
    "aggressive scan",
    "os fingerprinting",
    "port scan",
    "service detection",
    "service detection scan",
    "syn stealth scan",
    "vulnerability scan",
}

# Declaring the columns which hold the timestamp information. Keeping these columns in the data 
# is causing the model to refer to those as identical proxies causing model overfitting. 
# These leaky columns will be dropped as a part of preprocessing
LEAKY_COLUMNS = [
    # row identifiers
    "id", "expiration_id",
    # absolute timestamps — each capture file occupies a distinct time window
    "bidirectional_first_seen_ms", "bidirectional_last_seen_ms",
    "src2dst_first_seen_ms", "src2dst_last_seen_ms",
    "dst2src_first_seen_ms", "dst2src_last_seen_ms",
    # ephemeral / hardware identifiers
    "src_port",
    "src_ip", "src_mac", "dst_ip", "dst_mac",
    "src_oui", "dst_oui",
    # protocol-level fields with no variance / no semantic value
    "ip_version", "vlan_id", "tunnel_id",
]


def _class_aware_group_split(y_enc, groups, train_frac=0.60,
                             val_frac=0.20, test_frac=0.20, seed=42):
    """
    Class-aware group splitter. For each class, distribute its capture
    files (groups) across train / val / test so every class with >= 2
    files appears in both train and test. 
    """
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    y_enc  = np.asarray(y_enc)

    file_to_class = (
        pd.DataFrame({"file": groups, "cls": y_enc})
        .groupby("file")["cls"]
        .agg(lambda s: s.value_counts().idxmax())
    )

    train_files, val_files, test_files = [], [], []
    for cls in np.unique(y_enc):
        cls_files = file_to_class[file_to_class == cls].index.tolist()
        cls_files = list(rng.permutation(cls_files))
        n = len(cls_files)

        if n == 1:
            train_files += cls_files
        elif n == 2:
            train_files.append(cls_files[0])
            test_files.append(cls_files[1])
        elif n == 3:
            train_files.append(cls_files[0])
            val_files.append(cls_files[1])
            test_files.append(cls_files[2])
        else:  # n >= 4
            n_test  = max(1, int(round(n * test_frac)))
            n_val   = max(1, int(round(n * val_frac)))
            n_train = n - n_test - n_val
            if n_train < 1:
                n_val   = max(0, n_val + n_train - 1)
                n_train = n - n_test - n_val
            train_files += cls_files[:n_train]
            val_files   += cls_files[n_train:n_train + n_val]
            test_files  += cls_files[n_train + n_val:]

    train_set = set(train_files)
    val_set   = set(val_files)
    test_set  = set(test_files)

    train_idx = np.where(np.isin(groups, list(train_set)))[0]
    val_idx   = np.where(np.isin(groups, list(val_set)))[0]
    test_idx  = np.where(np.isin(groups, list(test_set)))[0]

    return train_idx, val_idx, test_idx


@dataclass
class SharedDomainData:
    raw_df:          Optional[pd.DataFrame] = None
    preprocessed_df: Optional[pd.DataFrame] = None

    X: Optional[pd.DataFrame] = None
    y: Optional[np.ndarray]   = None
    groups: Optional[np.ndarray] = None       # NEW — per-row capture file id

    combined_files: List[str] = field(default_factory=list)

    X_train: Optional[np.ndarray] = None
    X_val:   Optional[np.ndarray] = None
    X_test:  Optional[np.ndarray] = None
    y_train: Optional[np.ndarray] = None
    y_val:   Optional[np.ndarray] = None
    y_test:  Optional[np.ndarray] = None
    groups_train: Optional[np.ndarray] = None
    groups_val:   Optional[np.ndarray] = None
    groups_test:  Optional[np.ndarray] = None
    y_pred:  Optional[np.ndarray] = None
    y_proba: Optional[np.ndarray] = None

    preprocessing_log:     List[str] = field(default_factory=list)
    feature_selection_log: List[str] = field(default_factory=list)
    detection_log:         List[str] = field(default_factory=list)

    label_encoder     = None
    selected_features = []
    excluded_features = []
    model             = None
    scaler:           Optional[RobustScaler] = None
    report:           Optional[str]          = None

    TRAIN_FRAC            = 0.70
    VAL_FRAC              = 0.15
    TEST_FRAC             = 0.15
    RANDOM_STATE          = 42
    VARIANCE_THRESHOLD    = 0.01
    CORRELATION_THRESHOLD = 0.95
    CONFIDENCE_THRESHOLD  = 0.60


state = SharedDomainData()


## 3. Tool definitions

### 3.1 Data loading, cleaning, and preprocessing tools

In [5]:
@tool
def load_all_network_files(EVSE_A_file: str, EVSE_B_file: str):
    """Load EVSE-A & EVSE-B network csv file paths into a single list."""
    all_files = []
    all_files.extend(glob.glob(os.path.join(EVSE_A, "*.csv")))
    all_files.extend(glob.glob(os.path.join(EVSE_B, "*.csv")))
    state.preprocessing_log.append(
        f"Loaded files from {EVSE_A_file} & {EVSE_B_file}. "
        f"Total files combined are {len(all_files)}"
    )
    state.combined_files = all_files
    return f"Loaded {len(all_files)} files."


In [6]:
@tool
def create_consolidated_df_with_derived_labels():
    """
    Combine all per-file CSVs into a single dataframe.

    A Source_File column is preserved per row so we can later
    use GroupShuffleSplit and keep all flows from one capture file in the
    same split.
    """
    dfs = []
    for file in state.combined_files:
        df = pd.read_csv(file, low_memory=False)
        filename = file.lower().split("/")[-1].strip()
        df["Label"]        = 0 if "benign" in filename else 1
        df["State"]        = 1 if "charging" in filename else 0
        df["Attack_Label"] = " ".join(filename.split(".")[0].split("-")[2:])
        df["Source_File"]  = filename                    # NEW
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    os.makedirs(COMBINED_Directory, exist_ok=True)
    combined_df.to_csv(COMBINED_FILE_PATH, index=False)

    attack_dist  = combined_df["Attack_Label"].value_counts().to_dict()
    label_dist   = combined_df["Label"].value_counts().to_dict()
    missing_vals = int(combined_df.isnull().sum().sum())
    dtype_counts = {
        str(k): int(v) for k, v in
        combined_df.dtypes.value_counts().astype(str).items()
    }

    state.preprocessing_log += [
        f"shape {combined_df.shape}",
        f"dtype_distribution {dtype_counts}",
        f"label_distribution {label_dist}",
        f"attack_class_count {attack_dist}",
        f"total_missing_values {missing_vals}",
        f"unique_capture_files (groups) {combined_df['Source_File'].nunique()}",
    ]
    state.raw_df = combined_df
    return f"Loaded {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns."


In [7]:
@tool
def collapse_attack_labels():
    """
    Collapse the 41-class Attack_Label down to ~19 attack TYPES by stripping
    the EVSE-state prefix (charging / idle / maliciousev). The EVSE state is
    preserved separately in the State column and is excluded from features.
    The original 41-class label is retained in Attack_Label_Original for
    further analysis
    Attack_Label_Original is added to the EXCLUDE list inside
    apply_variance_threshold so it never reaches the model.

    Two filename-level inconsistencies are normalised so that synonymous labels collapse to the same class.
    """
    df = state.raw_df.copy()

    def _collapse(label: str) -> str:
        s = str(label).lower().strip()
        for prefix in STATE_PREFIXES:
            if s.startswith(prefix):
                s = s[len(prefix):]
                break
        return ATTACK_TYPE_NORMALISATIONS.get(s, s)

    df["Attack_Label_Original"] = df["Attack_Label"]
    df["Attack_Label"]          = df["Attack_Label"].apply(_collapse)

    n_before = df["Attack_Label_Original"].nunique()
    n_after  = df["Attack_Label"].nunique()

    new_dist        = df["Attack_Label"].value_counts().to_dict()
    files_per_class = (
        df.groupby("Attack_Label")["Source_File"].nunique().sort_values(ascending=False).to_dict()
    )

    state.raw_df = df
    state.preprocessing_log += [
        f"Collapsed Attack_Label: {n_before} → {n_after} attack types (state prefix stripped).",
        f"New attack-type row distribution: {new_dist}",
        f"Capture files per attack type: {files_per_class}",
    ]
    return f"Collapsed {n_before} labels into {n_after} attack types."


In [8]:
@tool
def aggregate_scan_labels():
    """
    Collapse the seven scan/reconnaissance attack types into a single
    'scan' class
    """
    df = state.raw_df.copy()

    n_before = df["Attack_Label"].nunique()
    df["Attack_Label"] = df["Attack_Label"].apply(
        lambda x: "scan" if x in SCAN_FAMILY else x
    )
    n_after = df["Attack_Label"].nunique()

    new_dist        = df["Attack_Label"].value_counts().to_dict()
    files_per_class = (
        df.groupby("Attack_Label")["Source_File"].nunique()
        .sort_values(ascending=False).to_dict()
    )

    state.raw_df = df
    state.preprocessing_log += [
        f"Aggregated scan family: {n_before} -> {n_after} attack types "
        f"({len(SCAN_FAMILY)} scan sub-types collapsed into 'scan').",
        f"Post-aggregation row distribution: {new_dist}",
        f"Post-aggregation capture files per class: {files_per_class}",
    ]
    return f"Aggregated {len(SCAN_FAMILY)} scan sub-types into 'scan' ({n_before} -> {n_after} classes)."


In [9]:
@tool
def handle_missing_values():
    """Drop columns whose missing-value rate exceeds 5%."""
    df = state.raw_df.copy()
    missing_pct = (df.isnull().mean() * 100).round(2)
    high_missing = missing_pct[missing_pct > 5].index.tolist()

    state.preprocessing_log.append(f"High-missing columns (>5%): {high_missing}")
    df = df.drop(columns=high_missing)
    state.raw_df = df
    state.preprocessing_log.append(
        f"Shape of dataframe after handling missing values {state.raw_df.shape}"
    )
    return f"Removed missing columns {list(high_missing)}"


In [10]:
@tool
def drop_leaky_columns():
    """
    NEW TOOL — drop columns that leak the label.
    These are timestamp / identifier columns that are perfectly correlated with
    the capture file and therefore with Attack_Label under random splitting.
    """
    df = state.raw_df.copy()
    present = [c for c in LEAKY_COLUMNS if c in df.columns]
    df = df.drop(columns=present)
    state.raw_df = df
    state.preprocessing_log.append(
        f"Dropped leaky/identifier columns ({len(present)}): {present}"
    )
    state.preprocessing_log.append(
        f"Shape of dataframe after dropping leaky columns {state.raw_df.shape}"
    )
    return f"Dropped {len(present)} leaky columns."


In [11]:
@tool
def handle_non_numeric_columns():
    """
    Encode the remaining low-cardinality categorical columns with one-hot.
    Source_File is preserved (it is needed for grouping but kept *outside* the
    feature matrix).
    """
    df = state.raw_df.copy()
    non_numeric_columns = list(
        df.select_dtypes(exclude=["int64", "float64"]).columns
    )
    state.preprocessing_log.append(
        f"Non numeric columns in the dataset {non_numeric_columns}"
    )

    # only encode true categorical traffic features
    categorical_columns = [
        c for c in ["application_name", "application_category_name"]
        if c in df.columns
    ]
    distribution_non_numeric = {
        col: df[col].nunique() for col in categorical_columns
    }
    state.preprocessing_log.append(
        f"One-hot encoding columns {categorical_columns}"
    )
    state.preprocessing_log.append(
        f"Distribution of non numeric columns in dataset {distribution_non_numeric}"
    )

    df = pd.get_dummies(df, columns=categorical_columns, dtype=int)

    state.preprocessed_df = df
    state.preprocessing_log.append(
        f"Shape after encoding non numerical columns {state.preprocessed_df.shape}"
    )
    return f"Shape after encoding {state.preprocessed_df.shape}"


### 3.2 Feature-selection tools

In [12]:
@tool
def apply_variance_threshold():
    """Remove features whose variance falls below state.VARIANCE_THRESHOLD."""
    EXCLUDE = ["Attack_Label", "Attack_Label_Original",
               "Label", "State", "Source_File"]
    df = state.preprocessed_df.copy()
    feature_cols = [c for c in df.columns if c not in EXCLUDE]

    X = df[feature_cols]
    state.feature_selection_log.append(f"Features before variance threshold: {X.shape[1]}")

    selector = VarianceThreshold(threshold=state.VARIANCE_THRESHOLD)
    selector.fit(X)

    kept = [c for c, k in zip(feature_cols, selector.get_support()) if k]
    dropped = [c for c, k in zip(feature_cols, selector.get_support()) if not k]

    state.selected_features = kept
    state.excluded_features = EXCLUDE

    state.feature_selection_log += [
        f"{len(dropped)} features dropped by variance threshold {state.VARIANCE_THRESHOLD}",
        f"Dropped features: {dropped}",
        f"Features after variance threshold: {len(kept)}",
    ]
    return f"Features after variance threshold: {len(kept)}"


In [13]:
@tool
def remove_correlated_features():
    """Drop one feature from each pair where |Pearson r| > CORRELATION_THRESHOLD."""
    df = state.preprocessed_df.copy()
    X = df[state.selected_features]
    y = df["Attack_Label"]
    groups = df["Source_File"].values

    SAMPLE_SIZE = min(200_000, len(X))
    X_sample = X.sample(n=SAMPLE_SIZE, random_state=state.RANDOM_STATE)

    corr_matrix = X_sample.corr().abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    to_drop = [c for c in upper.columns if any(upper[c] > state.CORRELATION_THRESHOLD)]

    remaining = [f for f in state.selected_features if f not in to_drop]
    state.selected_features = remaining

    state.X = df[remaining]
    state.y = y
    state.groups = groups

    state.feature_selection_log += [
        f"Features dropped by correlation filter {len(to_drop)}",
        f"Dropped correlated features: {to_drop}",
        f"Features after correlation filter: {len(remaining)}",
        f"Final shape: {state.X.shape}",
    ]
    return f"Final shape after correlation filter: {state.X.shape}"


### 3.3 Anomaly-class detection tools

In [14]:
@tool
def split_train_validation_test():
    """
    Class-aware group split (v4).

    GroupShuffleSplit is used that distributes class's capture files across train,
    validation, and test so that every class with >= 2 capture files
    appears in BOTH train and test (and in val whenever the class has
    >= 3 files).
    """
    X_all_df    = state.X.reset_index(drop=True).copy()
    y_all_raw   = state.y.reset_index(drop=True).copy()
    groups_all  = np.asarray(state.groups)

    # ---- Drop singleton-file classes ---------------------------------------
    file_counts_per_class = (
        pd.DataFrame({"lab": y_all_raw, "grp": groups_all})
        .groupby("lab")["grp"].nunique()
    )
    singleton_classes = file_counts_per_class[file_counts_per_class < 2].index.tolist()
    if singleton_classes:
        keep = ~y_all_raw.isin(singleton_classes)
        dropped_rows = int((~keep).sum())
        X_all_df   = X_all_df.loc[keep].reset_index(drop=True)
        y_all_raw  = y_all_raw.loc[keep].reset_index(drop=True)
        groups_all = groups_all[keep.values]
        state.detection_log.append(
            f"Dropped {len(singleton_classes)} singleton-file classes "
            f"({dropped_rows} rows): {singleton_classes}"
        )
    else:
        state.detection_log.append("No singleton-file classes to drop.")

    # ---- Encode labels -----------------------------------------------------
    le = LabelEncoder()
    y_enc = le.fit_transform(y_all_raw.astype(str))
    state.label_encoder = le
    n_classes = len(le.classes_)

    # ---- Class-aware group split ------------------------------------------
    train_idx, val_idx, test_idx = _class_aware_group_split(
        y_enc, groups_all,
        train_frac = 0.60, val_frac = 0.20, test_frac = 0.20,
        seed       = state.RANDOM_STATE,
    )

    train_classes = set(np.unique(y_enc[train_idx]))
    val_classes   = set(np.unique(y_enc[val_idx]))
    orphan_in_val = val_classes - train_classes
    if orphan_in_val:
        move_mask = np.isin(y_enc[val_idx], list(orphan_in_val))
        moved_idx = val_idx[move_mask]
        val_idx   = val_idx[~move_mask]
        train_idx = np.concatenate([train_idx, moved_idx])
        state.detection_log.append(
            f"Moved {len(moved_idx)} val rows -> train to cover "
            f"{len(orphan_in_val)} classes missing from train."
        )

    # ---- Materialise splits ------------------------------------------------
    X_train = X_all_df.iloc[train_idx]; y_train = y_enc[train_idx]; g_train = groups_all[train_idx]
    X_val   = X_all_df.iloc[val_idx];   y_val   = y_enc[val_idx];   g_val   = groups_all[val_idx]
    X_test  = X_all_df.iloc[test_idx];  y_test  = y_enc[test_idx];  g_test  = groups_all[test_idx]

    state.X_train, state.y_train, state.groups_train = X_train, y_train, g_train
    state.X_val,   state.y_val,   state.groups_val   = X_val,   y_val,   g_val
    state.X_test,  state.y_test,  state.groups_test  = X_test,  y_test,  g_test

    total = len(X_all_df)
    state.detection_log += [
        f" Train     : {X_train.shape[0]:,} rows "
        f"({X_train.shape[0] / total * 100:.1f}%) | "
        f"{len(np.unique(g_train))} capture files | "
        f"{len(np.unique(y_train))} classes",
        f" Validation: {X_val.shape[0]:,} rows "
        f"({X_val.shape[0] / total * 100:.1f}%) | "
        f"{len(np.unique(g_val))} capture files | "
        f"{len(np.unique(y_val))} classes",
        f" Test      : {X_test.shape[0]:,} rows "
        f"({X_test.shape[0] / total * 100:.1f}%) | "
        f"{len(np.unique(g_test))} capture files | "
        f"{len(np.unique(y_test))} classes",
        f" Classes encoded: {n_classes}",
    ]

    # Per-class file distribution (diagnostic)
    file_assign = (
        pd.DataFrame({"file": np.concatenate([g_train, g_val, g_test]),
                      "split": (["train"] * len(g_train) +
                                ["val"]   * len(g_val)   +
                                ["test"]  * len(g_test))})
        .drop_duplicates()
    )
    file_to_cls = pd.Series(y_enc, index=groups_all).groupby(level=0).first()
    file_assign["cls_idx"] = file_assign["file"].map(file_to_cls)
    file_assign["cls"]     = le.inverse_transform(file_assign["cls_idx"].astype(int))
    counts = (
        file_assign.groupby(["cls", "split"]).size().unstack(fill_value=0)
        .reindex(columns=["train", "val", "test"], fill_value=0)
        .sort_index()
    )
    state.detection_log.append("Per-class capture-file distribution:")
    state.detection_log.append(counts.to_string())

    return "Class-aware group split complete."


In [15]:
@tool
def normalize_features():
    """Fit RobustScaler on training rows only; transform val and test."""
    scaler = RobustScaler()
    state.X_train = scaler.fit_transform(state.X_train)
    state.X_val   = scaler.transform(state.X_val)
    state.X_test  = scaler.transform(state.X_test)
    state.scaler  = scaler
    state.detection_log.append("RobustScaler fitted on X_train and applied to X_val / X_test")
    return "Normalisation complete"


In [16]:
@tool
def train_attack_classifier():
    """
    Train LightGBM multi-class classifier with early stopping on the held-out
    *group-disjoint* validation set.
    """
    num_classes = len(state.label_encoder.classes_)

    state.model = lgb.LGBMClassifier(
        objective         = "multiclass",
        num_class         = num_classes,
        n_estimators      = 500,
        learning_rate     = 0.05,
        num_leaves        = 31,        # v4: was 63 — reduce session memorisation
        min_child_samples = 100,       # v4: was 20  — broader leaves
        class_weight      = None,      # v4: was "balanced" — see v4 patch note
        n_jobs            = 2,
        random_state      = state.RANDOM_STATE,
        verbose           = -1,
    )

    state.detection_log.append(
        f"Training LightGBM ({num_classes}-class) on "
        f"{state.X_train.shape[0]:,} samples × {state.X_train.shape[1]} features..."
    )

    # Defensive filter: drop any val rows whose class is still missing from train.
    # The split tool already guarantees this, but LightGBM's internal LabelEncoder
    # is strict and any slip-through would abort training.
    train_classes = set(np.unique(state.y_train))
    val_mask = np.isin(state.y_val, list(train_classes))
    if val_mask.sum() < len(state.y_val):
        dropped = len(state.y_val) - int(val_mask.sum())
        state.detection_log.append(
            f"Dropped {dropped} val rows with classes not in train "
            f"(defensive filter for LightGBM)."
        )
    X_val_safe = state.X_val[val_mask] if hasattr(state.X_val, '__getitem__') else state.X_val
    y_val_safe = state.y_val[val_mask]

    state.model.fit(
        state.X_train, state.y_train,
        eval_set=[(state.X_train, state.y_train), (X_val_safe, y_val_safe)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=25),
        ],
    )

    best_iter = state.model.best_iteration_
    state.detection_log += [
        f"Training complete. Best iteration: {best_iter}",
        "Model parameters: n_estimators=500, lr=0.05, num_leaves=31, "
        "min_child_samples=100, class_weight=None",
    ]
    return f"LightGBM training complete. Best iteration: {best_iter}. Classes: {num_classes}."


In [17]:
@tool
def evaluate_on_test_set():
    """
    Evaluate on the group-disjoint test set.

    Because some classes (e.g. 'idle benign', 'charging icmp flood') only
    have one or two capture files in the whole dataset, those classes may
    be entirely absent from the test split. The classification report uses
    `labels=` to constrain to classes present in y_test, which prevents
    misleading 0/0 rows.
    """
    y_pred = state.model.predict(state.X_test)
    state.y_pred = y_pred

    test_classes = np.unique(state.y_test)
    target_names = [state.label_encoder.classes_[i] for i in test_classes]

    report_dict = classification_report(
        state.y_test, y_pred,
        labels=test_classes, target_names=target_names,
        output_dict=True, zero_division=0,
    )
    report_str = classification_report(
        state.y_test, y_pred,
        labels=test_classes, target_names=target_names,
        zero_division=0,
    )

    weighted_f1 = report_dict["weighted avg"]["f1-score"]
    macro_f1    = report_dict["macro avg"]["f1-score"]
    accuracy    = report_dict["accuracy"]

    all_trained_classes = set(np.unique(state.y_train))
    missing_in_test     = sorted(all_trained_classes - set(test_classes))
    missing_names       = [state.label_encoder.classes_[i] for i in missing_in_test]

    state.detection_log += [
        f"Accuracy    : {accuracy:.4f}",
        f"Weighted F1 : {weighted_f1:.4f}",
        f"Macro F1    : {macro_f1:.4f}",
        f"Classes present in y_test : {len(test_classes)} / "
        f"{len(state.label_encoder.classes_)}",
        f"Classes trained but absent from test: "
        f"{missing_names if missing_names else 'none'}",
        f"Per-class report:\n{report_str}",
    ]

    low_recall = [
        cls for cls in target_names
        if report_dict.get(cls, {}).get("recall", 1.0) < 0.50
    ]
    state.detection_log.append(f"Attack classes with recall < 0.50: {low_recall}")
    return f"Evaluation complete. Accuracy: {accuracy:.4f} | Weighted F1: {weighted_f1:.4f}"


In [18]:
@tool
def apply_confidence_threshold():
    """Flag predictions whose top-class probability is below CONFIDENCE_THRESHOLD."""
    y_proba = state.model.predict_proba(state.X_test)
    max_proba = y_proba.max(axis=1)
    state.y_proba = y_proba

    low_mask = max_proba < state.CONFIDENCE_THRESHOLD
    total_low = int(low_mask.sum())
    pct_low = round(total_low / len(state.y_test) * 100, 2)

    state.detection_log += [
        f"Confidence threshold : {state.CONFIDENCE_THRESHOLD}",
        f"Low-confidence predictions : {total_low:,} ({pct_low}% of test set)",
    ]

    true_labels = state.label_encoder.inverse_transform(state.y_test)
    low_by_class = pd.Series(true_labels[low_mask]).value_counts().to_dict()
    state.detection_log.append(
        "Low-confidence breakdown by true class:\n"
        + "\n".join(f"  {cls:<45} {cnt:>6,}" for cls, cnt in low_by_class.items())
    )
    return (f"Threshold analysis complete. {total_low:,} predictions ({pct_low}%) "
            f"fell below confidence threshold of {state.CONFIDENCE_THRESHOLD}.")


### 3.4 Reporting tool

In [19]:
@tool
def generate_report() -> str:
    """Consolidate logs from all three agents into one report."""
    sep = "=" * 70
    def section(title, lines):
        return f"\n{sep}\n  {title}\n{sep}\n" + "\n".join(lines) + "\n"

    parts = [
        section("AGENT 1 — DATA PREPROCESSING",
                state.preprocessing_log or ["No log available."]),
        section("AGENT 2 — FEATURE SELECTION",
                state.feature_selection_log or ["No log available."]),
        section("AGENT 3 — ANOMALY CLASS DETECTION",
                state.detection_log or ["No log available."]),
    ]

    state.report = "\n".join(parts)
    return "Report generation successful"


## 4. Agent definitions

Note the **PreprocessingAgent** now calls a new `drop_leaky_columns` tool
before the categorical encoder. This is the critical change.


In [20]:
model = OpenAIChat(id="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.1)


In [21]:
preprocessing_agent = Agent(
    name="PreprocessingAgent",
    model=model,
    tools=[
        load_all_network_files,
        create_consolidated_df_with_derived_labels,
        collapse_attack_labels,        # v3 — strip EVSE-state prefix
        aggregate_scan_labels,         # v5 — collapse scan family into 'scan'
        handle_missing_values,
        drop_leaky_columns,
        handle_non_numeric_columns,
    ],
    instructions=[
        "You are a data preprocessing specialist for EVSE network data.",
        "Step 1: Call load_all_network_files to enumerate the raw CSVs.",
        "Step 2: Call create_consolidated_df_with_derived_labels to build the unified dataframe.",
        "Step 3: Call collapse_attack_labels to collapse the 41-class Attack_Label "
        "down to attack TYPES by stripping the EVSE-state prefix.",
        "Step 4: Call aggregate_scan_labels to collapse the seven scan/reconnaissance "
        "sub-types into a single 'scan' class.",
        "Step 5: Call handle_missing_values to drop columns with >5% missing values.",
        "Step 6: Call drop_leaky_columns to remove timestamp and identifier columns "
        "that would otherwise leak the label.",
        "Step 7: Call handle_non_numeric_columns to one-hot encode categorical features.",
        "Do not skip any tool. Report findings clearly after each tool call.",
    ],
    markdown=False,
)

feature_agent = Agent(
    name="FeatureSelectionAgent",
    model=model,
    tools=[apply_variance_threshold, remove_correlated_features],
    instructions=[
        "You are a feature selection specialist.",
        "Call apply_variance_threshold to remove features below the variance threshold.",
        "Call remove_correlated_features to drop highly correlated features.",
        "Do not skip any tool.",
    ],
    markdown=False,
)

detection_agent = Agent(
    name="DetectionAgent",
    model=model,
    tools=[
        split_train_validation_test,
        normalize_features,
        train_attack_classifier,
        evaluate_on_test_set,
        apply_confidence_threshold,
    ],
    instructions=[
        "You are an anomaly category prediction specialist for EVSE network traffic.",
        "Step 1: Call split_train_validation_test (group-aware, by capture file).",
        "Step 2: Call normalize_features.",
        "Step 3: Call train_attack_classifier.",
        "Step 4: Call evaluate_on_test_set.",
        "Step 5: Call apply_confidence_threshold.",
        "Do not skip any tool.",
    ],
    markdown=False,
)

reporting_agent = Agent(
    name="ReportingAgent",
    model=model,
    tools=[generate_report],
    instructions=[
        "You are a reporting specialist.",
        "Call generate_report to compile the pipeline summary.",
    ],
    markdown=False,
)


## 5. Run the pipeline

### 5.1 — Preprocessing agent

In [22]:
print("=" * 65)
print("STEP 1 — PreprocessingAgent")
print("=" * 65)

preprocessing_agent.print_response(
    f"Load the data files at locations '{EVSE_A}' & '{EVSE_B}'. "
    "Combine all the files into a single dataframe. "
    "Collapse the 41-class Attack_Label into attack types by stripping the EVSE-state prefix. "
    "Aggregate the scan-family sub-types into a single 'scan' class. "
    "Handle missing values. "
    "Drop leaky/identifier columns. "
    "Encode non-numeric columns."
)


STEP 1 — PreprocessingAgent


Output()

### 5.2 — Feature-selection agent

In [23]:
print("=" * 65)
print("STEP 2 — FeatureSelectionAgent")
print("=" * 65)

feature_agent.print_response(
    "Call apply_variance_threshold to drop low-variance features. "
    "Call remove_correlated_features to drop highly correlated features."
)


STEP 2 — FeatureSelectionAgent


Output()

### 5.3 — Detection agent

In [24]:
print("=" * 65)
print("STEP 3 — DetectionAgent")
print("=" * 65)

detection_agent.print_response(
    "Split the feature matrix into train/val/test sets using GROUPS by capture file. "
    "Scale the dataset by fitting the scaler only on training data. "
    "Train a multiclass classifier on the training data. "
    "Evaluate the classifier on the held-out test dataset. "
    "Evaluate predictions using a confidence threshold."
)


STEP 3 — DetectionAgent


Output()

[25]    training's multi_logloss: 0.241737      valid_1's multi_logloss: 3.79694

[50]    training's multi_logloss: 1.36838       valid_1's multi_logloss: 7.26525

[75]    training's multi_logloss: 0.20187       valid_1's multi_logloss: 2.30482

### 5.4 — Reporting agent

In [ ]:
print("=" * 65)
print("STEP 4 — ReportingAgent")
print("=" * 65)

reporting_agent.print_response(
    "Generate the full anomaly detection report for the EVSE network traffic data pipeline."
)


STEP 4 — ReportingAgent


Output()

In [ ]:
if state.report:
    print(state.report)
    with open("Anomaly_Detection_Network_Traffic_Report_Updated.txt", "w") as f:
        f.write(state.report)


## 6. Loss curve

In [ ]:
evals = state.model.evals_result_
train_loss = evals['training']['multi_logloss']
val_loss   = evals['valid_1']['multi_logloss']
best_iter  = state.model.best_iteration_

plt.figure(figsize=(10, 5))
plt.plot(train_loss, label='Training Loss',   color='steelblue', linewidth=1.5)
plt.plot(val_loss,   label='Validation Loss', color='tomato',    linewidth=1.5)
plt.axvline(x=best_iter, color='gray', linestyle='--', linewidth=1,
            label=f'Best Iteration: {best_iter}')
plt.xlabel('Boosting Round'); plt.ylabel('Multi-class Log Loss')
plt.title('LightGBM Training & Validation Loss Curve (group-aware split)')
plt.legend(); plt.grid(True, alpha=0.35); plt.tight_layout(); plt.show()


## 7. Confusion matrix (top 15 classes by support)

In [ ]:
y_pred = state.y_pred
class_names = state.label_encoder.classes_

top_n = 15
top_indices = pd.Series(state.y_test).value_counts().head(top_n).index.tolist()
top_labels  = [class_names[i] for i in top_indices]

mask = np.isin(state.y_test, top_indices)
y_true_top = state.y_test[mask]
y_pred_top = y_pred[mask]

cm = confusion_matrix(y_true_top, y_pred_top, labels=top_indices)
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top_labels, yticklabels=top_labels,
            linewidths=0.5, linecolor='lightgrey',
            cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.set_title(f'LightGBM Confusion Matrix — Top {top_n} Classes by Support')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout(); plt.show()


## 8. Macro-average ROC curve

In [ ]:
present = np.unique(state.y_test)
n_present = len(present)
y_test_bin = label_binarize(state.y_test, classes=list(present))
y_proba_present = state.y_proba[:, present]

fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(n_present):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(y_test_bin[:, i], y_proba_present[:, i])
    roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(n_present)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_present):
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= n_present
macro_auc = auc(all_fpr, mean_tpr)

plt.figure(figsize=(8, 6))
plt.plot(all_fpr, mean_tpr, label=f'Macro-average ROC (AUC = {macro_auc:.4f})',
         color='navy', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.fill_between(all_fpr, mean_tpr, alpha=0.08, color='navy')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'LightGBM ROC — Macro-average over {n_present} classes present in test')
plt.legend(loc='lower right'); plt.grid(True, alpha=0.35)
plt.tight_layout(); plt.show()
